Séance:S2
Nom:AGON KINHOU Fifamè Nathalie Ariane 
Porteur:P2
Etapes assignés:2.2 --> 2.5

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mrdwin/groupe5-allstate-appautomatque-l2-esgcvak/a_estimer.csv
/kaggle/input/datasets/mrdwin/groupe5-allstate-appautomatque-l2-esgcvak/train.csv


In [2]:
#La barre de bruit
import pandas as pd

DOSSIER = "/kaggle/input/datasets/mrdwin/groupe5-allstate-appautomatque-l2-esgcvak/"

train = pd.read_csv(DOSSIER + "train.csv")
test  = pd.read_csv(DOSSIER + "a_estimer.csv")

print(train.shape, test.shape)    # attendu : (150654, 132) (37664, 131)
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error as mae

y = train["loss"].values
kf = KFold(n_splits=5, shuffle=True, random_state=0)

scores = []
for tr_idx, va_idx in kf.split(train):
    c = np.median(y[tr_idx])                 # constante calculée sur le train du pli
    scores.append(mae(y[va_idx], [c] * len(va_idx)))

scores = np.array(scores)
print("MAE par pli :", scores.round(2))
print("moyenne :", scores.mean().round(2))
print("écart-type (la barre de bruit) :", scores.std(ddof=1).round(2))

(150654, 132) (37664, 131)
MAE par pli : [1806.01 1828.11 1831.3  1789.09 1798.01]
moyenne : 1810.51
écart-type (la barre de bruit) : 18.56


In [3]:
#Le premier modèle
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingRegressor

cat_cols  = [c for c in train.columns if c.startswith("cat")]
cont_cols = [c for c in train.columns if c.startswith("cont")]

prep = ColumnTransformer([
    ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_cols),
    ("cont", "passthrough", cont_cols),
])

modele = Pipeline([
    ("prep", prep),
    ("hgb", HistGradientBoostingRegressor(loss="absolute_error", random_state=0)),
])

X = train[cat_cols + cont_cols]
scores_m = []
for tr_idx, va_idx in kf.split(X):
    modele.fit(X.iloc[tr_idx], y[tr_idx])
    scores_m.append(mae(y[va_idx], modele.predict(X.iloc[va_idx])))

scores_m = np.array(scores_m)
print("MAE du modèle par pli :", scores_m.round(2))
print("moyenne :", scores_m.mean().round(2), "±", scores_m.std(ddof=1).round(2))
print("gain vs plancher :", (scores.mean() - scores_m.mean()).round(2))

MAE du modèle par pli : [1164.28 1174.69 1179.81 1158.52 1168.6 ]
moyenne : 1169.18 ± 8.38
gain vs plancher : 641.32
